# Sort, look up, answer
A shop has a shipping note. First use plain Python; then (optionally) see the same steps as a LangGraph graph. Core path needs no API key, model or framework. Read [the lesson](../../docs/08-langchain-langgraph-langsmith.md) first.
Predict: what should happen if the question has no word like 'ship'?

In [ ]:
def route(question: str) -> dict:
    topic = 'shipping' if 'ship' in question.lower() else 'unknown'
    answer = 'Shipping takes three days.' if topic == 'shipping' else 'No matching policy.'
    return {'question': question, 'topic': topic, 'answer': answer}
assert route('When does it ship?')['topic'] == 'shipping'
assert route('Unknown')['topic'] == 'unknown'
print(route('When does it ship?'))
print(route('When do you open?'))

A graph is this same sequence expressed as named steps, called nodes. The 'state' is the dictionary containing the question, topic and answer. Run the next cell only if you installed the optional agents dependencies.

In [ ]:
try:
    from langgraph.graph import END, START, StateGraph
except ImportError:
    print('LangGraph not installed; plain Python route above is complete.')
else:
    from typing import TypedDict
    class State(TypedDict):
        question: str
        topic: str
        answer: str
    def classify(state: State) -> dict:
        return {'topic': 'shipping' if 'ship' in state['question'].lower() else 'unknown'}
    def answer(state: State) -> dict:
        return {'answer': 'Shipping takes three days.' if state['topic'] == 'shipping' else 'No matching policy.'}
    graph = StateGraph(State)
    graph.add_node('classify', classify)
    graph.add_node('answer', answer)
    graph.add_edge(START, 'classify')
    graph.add_edge('classify', 'answer')
    graph.add_edge('answer', END)
    result = graph.compile().invoke({'question': 'When does it ship?', 'topic': '', 'answer': ''})
    assert result['answer'] == route(result['question'])['answer']
    print(result)

## Try it
Change the shipping sentence in both functions. Why did you have to edit two places? This is why applications should share the policy data rather than duplicate it. Add a third topic only after both implementations work for the first two cases.